Amazon Beauty dataset build and Item prompt generation notebook.

**Preprocessing overview (run this notebook from `build_datasets_and_prompts` directory):**
- Download Beauty review and metadata files if not loaded.
- Apply 5-core filtering and build chronologically sorted `(user_id, item_id, time)` interactions.
- Reindex users/items to contiguous integer IDs and save interaction files.
- Generate **Item prompts**: JSON-structured answers (item summary + potential user analysis) for LLM calls.

**Main outputs under `./data/Beauty/`:**
- `inter.csv`, `meta.csv`, `Beauty.txt`
- `user2id.json`, `item2id.json`, `item_prompt_input.pkl`


In [ ]:
import os
import gzip
import ast
import json
import random
import pickle
import subprocess
import pandas as pd
from datetime import datetime, timezone

In [ ]:
def parse(path):
    g = gzip.open(path, 'rb')
    for l in g:
        yield eval(l)


def get_df(path):
    i = 0
    df = {}

    for d in parse(path):
        df[i] = d
        i += 1

    return pd.DataFrame.from_dict(df, orient='index')

In [ ]:
DATASET = 'Beauty'
RAW_PATH = os.path.join('./data/', DATASET)
DATA_FILE = 'reviews_{}_5.json.gz'.format(DATASET)
META_FILE = 'meta_{}.json.gz'.format(DATASET)

In [ ]:
# download data if not exists
if not os.path.exists(RAW_PATH):
    subprocess.call('mkdir ' + RAW_PATH, shell=True)

if not os.path.exists(os.path.join(RAW_PATH, DATA_FILE)):
    print('Downloading interaction data into ' + RAW_PATH)
    subprocess.call(
        'cd {} && curl -O http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_{}_5.json.gz'
        .format(RAW_PATH, DATASET), shell=True)

if not os.path.exists(os.path.join(RAW_PATH, META_FILE)):
    print('Downloading item metadata into ' + RAW_PATH)
    subprocess.call(
        'cd {} && curl -O http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/meta_{}.json.gz'
        .format(RAW_PATH, DATASET), shell=True)

In [ ]:
raw_interactions_df = get_df(os.path.join(RAW_PATH, DATA_FILE))
raw_interactions_df.head()

In [ ]:
meta_df = get_df(os.path.join(RAW_PATH, META_FILE))
meta_df.head()

In [ ]:
# 5-core filtering (each user/item appears at least 5 times)
def filter_df(df):
    while True:
        asin_counts = df['asin'].value_counts()
        reviewerID_counts = df['reviewerID'].value_counts()

        asin_to_remove = asin_counts[asin_counts < 5].index
        reviewerID_to_remove = reviewerID_counts[reviewerID_counts < 5].index

        if len(asin_to_remove) == 0 and len(reviewerID_to_remove) == 0:
            break

        df = df[~df['asin'].isin(asin_to_remove)]
        df = df[~df['reviewerID'].isin(reviewerID_to_remove)]

    return df


n_users = raw_interactions_df['reviewerID'].value_counts().size
n_items = raw_interactions_df['asin'].value_counts().size
n_clicks = len(raw_interactions_df)
min_time = raw_interactions_df['unixReviewTime'].min()
max_time = raw_interactions_df['unixReviewTime'].max()
time_format = '%Y-%m-%d'

print("========before filtering=============")
print('# Users:', n_users)
print('# Items:', n_items)
print('# Interactions:', n_clicks)
print('Time Span: {}/{}'.format(
    datetime.fromtimestamp(min_time, timezone.utc).strftime(time_format),
    datetime.fromtimestamp(max_time, timezone.utc).strftime(time_format))
)

print("\n========after filtering=============")
data_df = filter_df(raw_interactions_df)
n_users = data_df['reviewerID'].value_counts().size
n_items = data_df['asin'].value_counts().size
n_clicks = len(data_df)
min_time = data_df['unixReviewTime'].min()
max_time = data_df['unixReviewTime'].max()

print('# Users:', n_users)
print('# Items:', n_items)
print('# Interactions:', n_clicks)
print('Time Span: {}/{}'.format(
    datetime.fromtimestamp(min_time, timezone.utc).strftime(time_format),
    datetime.fromtimestamp(max_time, timezone.utc).strftime(time_format))
)

In [ ]:
# Only retain items that appear in interaction data
useful_meta_df = meta_df[meta_df['asin'].isin(data_df['asin'])].reset_index(drop=True)
((useful_meta_df.isnull().sum()) / useful_meta_df.shape[0]).sort_values(ascending=False).map(lambda x:"{:.2%}".format(x))

# Build Dataset

### Interaction data

In [ ]:
out_df = data_df.rename(columns={'asin': 'item_id', 'reviewerID': 'user_id', 'unixReviewTime': 'time'})
out_df = out_df[['user_id', 'item_id', 'time']]
out_df = out_df.drop_duplicates(['user_id', 'item_id', 'time'])
out_df = out_df.sort_values(by=['user_id', 'time'], kind='mergesort').reset_index(drop=True)
out_df.head()

In [ ]:
# reindex (start from 1)
uids = out_df['user_id'].unique()
user2id = dict(zip(uids, range(1, len(uids) + 1)))
iids = out_df['item_id'].unique()
item2id = dict(zip(iids, range(1, len(iids) + 1)))

out_df['user_id'] = out_df['user_id'].apply(lambda x: user2id[x])
out_df['item_id'] = out_df['item_id'].apply(lambda x: item2id[x])
out_df.head()

In [ ]:
# save data
out_df.to_csv(RAW_PATH + '/inter.csv', index=False)
useful_meta_df['item_id'] = useful_meta_df['asin'].apply(lambda x: item2id[x])
useful_meta_df.to_csv(RAW_PATH + '/meta.csv', index=False)

# save id mappings (for downstream reproducibility)
with open(RAW_PATH + '/user2id.json', 'w') as f:
    json.dump(user2id, f)
with open(RAW_PATH + '/item2id.json', 'w') as f:
    json.dump(item2id, f)

In [ ]:
# Export interactions for downstream recommenders (space-separated, no header)
out_df.drop(columns=['time']).to_csv(
    RAW_PATH + '/Beauty.txt', sep=' ', index=False, header=False
)

In [ ]:
inter = out_df.copy()
# train-only: sort by time, drop last 2 interactions per user (val/test)
inter = inter.sort_values(by=['user_id', 'time'], kind='mergesort').reset_index(drop=True)
_pos = inter.groupby('user_id').cumcount()
_n = inter.groupby('user_id')['user_id'].transform('size')
inter = inter[_pos < (_n - 2)].reset_index(drop=True)

In [ ]:
item_prompt_meta_df = useful_meta_df.copy()
# Only keep columns needed for item_info (+ item_id key)
item_info_columns = ['item_id', 'title', 'description', 'price', 'categories']

item_prompt_meta_df = item_prompt_meta_df[item_info_columns].copy()
item_prompt_meta_df['title'] = item_prompt_meta_df['title'].fillna('missing or unknown')
item_prompt_meta_df['description'] = item_prompt_meta_df['description'].fillna('missing or unknown')
item_prompt_meta_df['price'] = item_prompt_meta_df['price'].fillna('missing or unknown')
item_prompt_meta_df['categories'] = item_prompt_meta_df['categories'].fillna('missing or unknown')

In [ ]:
def process_categories(cat_val):
    """Normalize categories to a readable path string."""

    # Common null-like cases
    if cat_val is None:
        return 'missing or unknown'

    # If the value is already a parsed list/tuple (happens when using `meta_df` directly)
    if isinstance(cat_val, (list, tuple)):
        if len(cat_val) == 0:
            return 'missing or unknown'

        first = cat_val[0]
        if isinstance(first, (list, tuple)):
            return ' > '.join(map(str, first)) if len(first) > 0 else 'missing or unknown'

        return ' > '.join(map(str, cat_val))

    # Scalar / string cases
    if isinstance(cat_val, float) and pd.isna(cat_val):
        return 'missing or unknown'

    s = str(cat_val)
    if s == '' or s == 'nan':
        return 'missing or unknown'

    try:
        cat_list = ast.literal_eval(s)
        if isinstance(cat_list, list) and len(cat_list) > 0:
            first = cat_list[0]

            if isinstance(first, (list, tuple)):
                return ' > '.join(map(str, first)) if len(first) > 0 else 'missing or unknown'

            return ' > '.join(map(str, cat_list))
            
        return 'missing or unknown'
        
    except (ValueError, SyntaxError, IndexError, TypeError):
        return 'missing or unknown'


item_prompt_meta_df['categories'] = item_prompt_meta_df['categories'].apply(process_categories)
meta_dict = item_prompt_meta_df.set_index('item_id').T.to_dict()

In [ ]:
# seq slide augmentation
def prepare_data_augmentation(df):
    max_item_list_len = 20
    last_uid = None
    uid_list, item_list, target, item_list_length = [], [], [], []

    for _, row in df.iterrows():
        uid, item_id = row['user_id'], row['item_id']

        if last_uid != uid:
            last_uid = uid
            seq = []

        else:
            if len(seq) > max_item_list_len:
                seq = seq[1:]

            uid_list.append(uid)
            item_list.append(seq[:])
            target.append(item_id)
            item_list_length.append(len(seq))

        seq.append(item_id)

    return uid_list, item_list, target, item_list_length


uid_list, item_list, target, item_list_length = prepare_data_augmentation(inter)

In [ ]:
filtered_list = [sublist[-11:] for sublist in item_list if (len(sublist) >= 2 and len(sublist) <= 21)]
print("length of filtered_list", len(filtered_list))

last_values = set(sublist[-1] for sublist in filtered_list)
end_dict = {value: [] for value in last_values}

for sublist in filtered_list:
    last_value = sublist[-1]
    end_dict[last_value].append(sublist)

random.seed(2026)

for key in end_dict:
    if len(end_dict[key]) > 5:
        end_dict[key] = random.sample(end_dict[key], 5)

end_dict_text = {}
for key in end_dict:
    end_dict_text[key] = []

    for ls in end_dict[key]:
        ls_text = [meta_dict[item]['title'] for item in ls]
        ls_text = [val for val in ls_text if val != "missing or unknown"]
        end_dict_text[key].append(ls_text)

end_dict_text_formatted = {}
for item_id in end_dict_text.keys():
    same_target_seqs = ''

    for seq in end_dict_text[item_id]:
        seq = ' -> '.join(seq)
        seq = '[' + seq + '] # '
        seq += " \n "
        same_target_seqs += seq

    end_dict_text_formatted[item_id] = same_target_seqs

In [ ]:
def item_template(item_dict):
    title = item_dict.get('title', 'missing or unknown')
    if pd.isna(title) or title == '' or str(title) == 'nan':
        title = 'missing or unknown'

    categories = item_dict.get('categories', 'missing or unknown')
    if pd.isna(categories) or categories == '' or str(categories) == 'nan':
        categories = 'missing or unknown'

    desc = item_dict.get('description', 'missing or unknown')
    if pd.isna(desc) or desc == '' or str(desc) == 'nan':
        desc = 'missing or unknown'

    price = item_dict.get('price', 'missing or unknown')
    if pd.isna(price) or price == '' or str(price) == 'nan':
        price = 'missing or unknown'

    text = (
        f"The product name is {title}; "
        f"categories are {categories}; "
        f"price is {price}; "
        f"The detailed description of the product is {str(desc)[:1000]}..."
    )

    return text


item_meta_formatted = {}
for item_id in item2id.values():
    item_meta_formatted[item_id] = item_template(meta_dict[item_id])

In [ ]:
def complete_prompt_gen(item_info, history):
    prompt = f"""
    Assume you are a beauty and personal care recommendation expert. Please help me analyze a specific beauty product. You will be provided with the following information:
    1) The attributes of the beauty product: {item_info};
    2) The historical purchase information of users who have bought this product: {history}. Here, different sequences are separated by '#', and each sequence is in LIST format, representing a certain user's historical purchases. Items in each sequence are separated by '->', and the last item in all sequences is the specific product mentioned above.
    
    Requirements:
    1) Please briefly describe the given target item.
    2) Based on the provided sequences, please analyze what type of users would purchase this item. Please do not generally say "beauty lovers" or generic personal-care users, as all users in this context have a need to purchase beauty products. Please provide a more detailed granularity.
    Please provide your answer in JSON format, following this structure:
    {{
    "item summary": "A description of the item, no more than 80 words.", 
    "potential user analysis": "what type of users would purchase this item, no more than 50 words."
    }}
    """

    return prompt


item_prompt = {}
for item_id in item2id.values():
    item_info = item_meta_formatted[item_id]

    if item_id in end_dict_text_formatted.keys():
        history = end_dict_text_formatted[item_id]
    else:
        history = ' [None] '
        
    item_prompt[item_id] = complete_prompt_gen(item_info, history)

In [ ]:
with open(RAW_PATH + '/item_prompt_input.pkl', 'wb') as pickle_file:
    pickle.dump(item_prompt, pickle_file)

In [ ]:
# Preview: first 5 item prompts (by item_id)
with open(RAW_PATH + '/item_prompt_input.pkl', 'rb') as f:
    _item_prompt_preview = pickle.load(f)

for _k in sorted(_item_prompt_preview.keys())[:5]:
    print('=' * 80)
    print(f'item_id: {_k}')
    print('=' * 80)
    print(_item_prompt_preview[_k])
    print()